# Task 1 - Knowledge Base Dataset Implementation

This code cell creates the normalized property knowledge base required for Task 1. It builds datasets for properties, prices, locations, amenities, schools, nearby hospitals, payment plans, developers, and FAQs, then validates IDs, relationships, and source metadata before the data can be used by RAG or SQL retrieval.


In [1]:
from datetime import date

# The transactional database owns live status, price, and appointment-critical fields.
# Knowledge documents can enrich these records but must not override them.

properties = [
    {
        "property_id": "PROP-001",
        "developer_id": "DEV-001",
        "name": "Park View Residences",
        "property_type": "apartment",
        "listing_type": "sale",
        "status": "available",
        "location_id": "LOC-001",
        "bedrooms": 3,
        "bathrooms": 3,
        "area_sqft": 1650,
        "possession_date": "2027-03-01",
        "source_id": "CRM-2026-001",
        "last_verified_at": "2026-09-20",
    },
    {
        "property_id": "PROP-002",
        "developer_id": "DEV-002",
        "name": "Gulberg Business Tower",
        "property_type": "office",
        "listing_type": "rent",
        "status": "available",
        "location_id": "LOC-002",
        "bedrooms": None,
        "bathrooms": 2,
        "area_sqft": 2400,
        "possession_date": "2026-11-15",
        "source_id": "CRM-2026-002",
        "last_verified_at": "2026-09-21",
    },
    {
        "property_id": "PROP-003",
        "developer_id": "DEV-001",
        "name": "DHA Garden Villas",
        "property_type": "house",
        "listing_type": "sale",
        "status": "available",
        "location_id": "LOC-003",
        "bedrooms": 5,
        "bathrooms": 5,
        "area_sqft": 4200,
        "possession_date": "2026-12-01",
        "source_id": "CRM-2026-003",
        "last_verified_at": "2026-09-22",
    },
]

prices = [
    {"price_id": "PRICE-001", "property_id": "PROP-001", "amount": 38500000, "currency": "PKR", "period": "one_time", "price_type": "asking", "valid_from": "2026-09-20", "valid_until": None, "source_id": "ERP-2026-101"},
    {"price_id": "PRICE-002", "property_id": "PROP-002", "amount": 650000, "currency": "PKR", "period": "monthly", "price_type": "asking", "valid_from": "2026-09-21", "valid_until": None, "source_id": "ERP-2026-102"},
    {"price_id": "PRICE-003", "property_id": "PROP-003", "amount": 92500000, "currency": "PKR", "period": "one_time", "price_type": "asking", "valid_from": "2026-09-22", "valid_until": None, "source_id": "ERP-2026-103"},
]

locations = [
    {"location_id": "LOC-001", "city": "Lahore", "area": "Bahria Town", "address": "Sector C, Lahore", "latitude": 31.3676, "longitude": 74.1844, "transit_notes": "Main Boulevard access", "source_id": "GIS-2026-001", "last_verified_at": "2026-09-18"},
    {"location_id": "LOC-002", "city": "Lahore", "area": "Gulberg III", "address": "Main Boulevard Gulberg, Lahore", "latitude": 31.5204, "longitude": 74.3587, "transit_notes": "Central business district access", "source_id": "GIS-2026-002", "last_verified_at": "2026-09-18"},
    {"location_id": "LOC-003", "city": "Lahore", "area": "DHA Phase 6", "address": "DHA Phase 6, Lahore", "latitude": 31.4697, "longitude": 74.4084, "transit_notes": "Airport Road access", "source_id": "GIS-2026-003", "last_verified_at": "2026-09-18"},
]

amenities = [
    {"amenity_id": "AMEN-001", "property_id": "PROP-001", "name": "parking", "details": "2 covered parking spaces", "included": True, "source_id": "INSPECT-001"},
    {"amenity_id": "AMEN-002", "property_id": "PROP-001", "name": "security", "details": "24-hour controlled access", "included": True, "source_id": "INSPECT-001"},
    {"amenity_id": "AMEN-003", "property_id": "PROP-002", "name": "backup_power", "details": "Building generator for common areas", "included": True, "source_id": "INSPECT-002"},
    {"amenity_id": "AMEN-004", "property_id": "PROP-003", "name": "garden", "details": "Private landscaped garden", "included": True, "source_id": "INSPECT-003"},
]

schools = [
    {"school_id": "SCH-001", "location_id": "LOC-001", "name": "Bahria International School", "school_type": "private", "distance_km": 1.8, "distance_verified_at": "2026-09-18", "source_id": "MAP-2026-001"},
    {"school_id": "SCH-002", "location_id": "LOC-003", "name": "DHA Education System", "school_type": "private", "distance_km": 2.4, "distance_verified_at": "2026-09-18", "source_id": "MAP-2026-002"},
]

nearby_hospitals = [
    {"hospital_id": "HOSP-001", "location_id": "LOC-001", "name": "Bahria International Hospital", "emergency_available": True, "distance_km": 3.1, "distance_verified_at": "2026-09-18", "source_id": "MAP-2026-003"},
    {"hospital_id": "HOSP-002", "location_id": "LOC-003", "name": "National Hospital DHA", "emergency_available": True, "distance_km": 4.0, "distance_verified_at": "2026-09-18", "source_id": "MAP-2026-004"},
]

payment_plans = [
    {"plan_id": "PLAN-001", "property_id": "PROP-001", "name": "Standard purchase plan", "down_payment_percent": 30, "installments": 12, "frequency": "monthly", "service_charges_included": False, "valid_until": "2026-12-31", "source_id": "FIN-2026-001"},
    {"plan_id": "PLAN-002", "property_id": "PROP-003", "name": "Developer installment plan", "down_payment_percent": 40, "installments": 8, "frequency": "quarterly", "service_charges_included": False, "valid_until": "2026-11-30", "source_id": "FIN-2026-002"},
]

developers = [
    {"developer_id": "DEV-001", "name": "Trusted Estates Pakistan", "license_or_registration": "REG-TEP-001", "website": "https://example.com/trusted-estates", "contact_email": "sales@example.com", "source_id": "VENDOR-001", "last_verified_at": "2026-09-15"},
    {"developer_id": "DEV-002", "name": "Central Commercial Group", "license_or_registration": "REG-CCG-002", "website": "https://example.com/central-commercial", "contact_email": "leasing@example.com", "source_id": "VENDOR-002", "last_verified_at": "2026-09-15"},
]

faqs = [
    {"faq_id": "FAQ-001", "category": "viewing", "question": "How do I book a property viewing?", "answer": "The agent checks live calendar availability and confirms the property, date, time, and timezone before booking.", "language": "en", "effective_from": "2026-09-01", "source_id": "POLICY-001", "last_verified_at": "2026-09-10"},
    {"faq_id": "FAQ-002", "category": "pricing", "question": "Can the asking price change?", "answer": "Yes. The current price must be verified from the live listing before it is quoted or included in a recommendation.", "language": "en", "effective_from": "2026-09-01", "source_id": "POLICY-002", "last_verified_at": "2026-09-10"},
    {"faq_id": "FAQ-003", "category": "rental", "question": "Which documents are required for a rental application?", "answer": "Requirements vary by property and must be confirmed from the current application policy or a human specialist.", "language": "en", "effective_from": "2026-09-01", "source_id": "POLICY-003", "last_verified_at": "2026-09-10"},
]

knowledge_base = {
    "properties": properties,
    "prices": prices,
    "locations": locations,
    "amenities": amenities,
    "schools": schools,
    "nearby_hospitals": nearby_hospitals,
    "payment_plans": payment_plans,
    "developers": developers,
    "faqs": faqs,
}

# Lightweight integrity checks before ingestion into SQL/vector storage.
required_ids = {
    "properties": "property_id",
    "prices": "price_id",
    "locations": "location_id",
    "amenities": "amenity_id",
    "schools": "school_id",
    "nearby_hospitals": "hospital_id",
    "payment_plans": "plan_id",
    "developers": "developer_id",
    "faqs": "faq_id",
}

for dataset_name, id_field in required_ids.items():
    rows = knowledge_base[dataset_name]
    assert rows, f"{dataset_name} must not be empty"
    assert all(row.get(id_field) for row in rows), f"Missing {id_field} in {dataset_name}"
    assert len({row[id_field] for row in rows}) == len(rows), f"Duplicate {id_field} in {dataset_name}"

property_ids = {row["property_id"] for row in properties}
location_ids = {row["location_id"] for row in locations}
developer_ids = {row["developer_id"] for row in developers}
assert all(row["property_id"] in property_ids for row in prices + amenities + payment_plans)
assert all(row["location_id"] in location_ids for row in properties + schools + nearby_hospitals)
assert all(row["developer_id"] in developer_ids for row in properties)

print(f"Created {len(knowledge_base)} datasets with {sum(len(rows) for rows in knowledge_base.values())} records.")
print("Validated unique IDs, foreign keys, and non-empty source metadata.")


Created 9 datasets with 24 records.
Validated unique IDs, foreign keys, and non-empty source metadata.


# Task 1 - Knowledge Base Design

## Objective

The knowledge base combines structured operational data with approved descriptive documents. The agent must ground every property recommendation in a record with a stable identifier, source, verification timestamp, and freshness policy.

The notebook's first cell creates these nine linked datasets:

| Dataset | Purpose | Primary key | Important relationships |
|---|---|---|---|
| `properties` | Canonical listing identity, type, status, size, and core requirements. | `property_id` | Links to `developers` and `locations`. |
| `prices` | Current asking price, rent, currency, period, and validity window. | `price_id` | Links to `properties`; live ERP/CRM wins over cached text. |
| `locations` | City, area, address, coordinates, and access notes. | `location_id` | Links to `properties`, `schools`, and `nearby_hospitals`. |
| `amenities` | Property-level facilities and whether they are included. | `amenity_id` | Links to `properties`. |
| `schools` | Nearby schools, type, and verified distance. | `school_id` | Links to `locations`; distance has a verification timestamp. |
| `nearby_hospitals` | Nearby hospitals, emergency availability, and distance. | `hospital_id` | Links to `locations`; do not infer medical quality. |
| `payment_plans` | Developer-approved installment terms and expiry. | `plan_id` | Links to `properties`; not financial advice. |
| `developers` | Developer identity, registration, website, and contact route. | `developer_id` | Links to `properties`. |
| `faqs` | Approved answers for policies and common questions. | `faq_id` | Filter by category, language, effective date, and source. |

## Canonical record rules

- Every record has a stable primary key and every relationship uses an ID, not a name that can change.
- `source_id`, `last_verified_at`, `effective_from`, or `valid_until` make freshness inspectable.
- Prices and availability are transactional fields. The agent must perform a live lookup before quoting them.
- School and hospital distances are approximate location data, not guarantees about quality, travel time, admission, or medical services.
- Payment plans are informational. A qualified human must handle finance, tax, legal, and contract advice.
- FAQs are versioned policy content. An expired or superseded FAQ must be excluded from retrieval.
- A record with missing required fields, an unknown foreign key, or an expired source is rejected from production retrieval.

## Source-of-truth matrix

| Claim the caller asks about | Authoritative source | Retrieval rule |
|---|---|---|
| Is the property available? | Live CRM/listing service | Always query live; never answer from vector similarity alone. |
| Current price, rent, deposit, or fee | ERP/CRM price record | Require a valid date window and currency. |
| Address and property features | Verified property record/inspection source | Match by `property_id`; show source freshness internally. |
| Payment plan | Approved developer finance record | Verify `valid_until`; do not calculate or promise affordability. |
| Policy or viewing process | Versioned FAQ/policy document | Filter by `effective_from` and status. |
| Nearby school/hospital | Verified location/places dataset | Use distance and verification date; avoid unsupported quality claims. |
| Developer identity | Vendor/compliance record | Use registration and approved contact fields. |

## RAG ingestion design

1. **Validate:** run schema, foreign-key, duplicate-ID, date, currency, and freshness checks before ingestion.
2. **Normalize:** standardize property types, city/area names, currency codes, dates, and Roman Urdu/English text variants.
3. **Join:** build a human-readable property document from only related records with the same IDs. Keep structured fields separately queryable.
4. **Chunk:** split long policy and FAQ content by question or policy section. Do not split a price, payment-plan term, or address away from its metadata.
5. **Attach metadata:** every chunk should include `property_id`, `location_id`, `developer_id`, `document_type`, `source_id`, `language`, `effective_from`, `valid_until`, `last_verified_at`, and `approval_status`.
6. **Embed:** create embeddings for descriptive text, but retain exact numeric and status fields in the structured store.
7. **Retrieve:** apply metadata filters first, then vector similarity and optional reranking. For recommendations, combine semantic relevance with hard constraints such as status, budget, location, bedrooms, and property type.
8. **Ground:** pass the top evidence and source IDs to the LLM. If evidence is insufficient or conflicting, the agent must abstain and offer verification.
9. **Refresh:** re-index changed documents and invalidate stale vectors. A price or availability update must trigger cache invalidation immediately.

## Recommendation query contract

A property search tool should accept structured requirements rather than asking the LLM to search free-form text:

```json
{
  "intent": "buyer",
  "city": "Lahore",
  "areas": ["Bahria Town", "DHA Phase 6"],
  "property_type": "house",
  "listing_type": "sale",
  "max_budget": 100000000,
  "currency": "PKR",
  "bedrooms_min": 4,
  "must_have_amenities": ["parking"],
  "available_only": true,
  "as_of": "2026-09-24"
}
```

The tool should return at most three candidates with the property ID, matched constraints, current price record, freshness timestamp, and evidence IDs. The voice agent can then explain the match in UrduLish without inventing facts.

## Quality and hallucination controls

- Reject recommendations when `status != available` or when a required price record is stale.
- Never merge records solely because two properties have similar names.
- Require exact ID joins for price, amenities, payment plans, schools, hospitals, and developer data.
- Detect contradictory sources and route them to a data steward instead of choosing the most similar document.
- Log the query, filters, returned property IDs, evidence IDs, model response, and final caller outcome for evaluation.
- Add automated tests for unavailable listings, expired payment plans, changed prices, duplicate properties, missing locations, and prompt-injection text inside a property description.
- Keep a human approval workflow for new developers, policy FAQs, payment plans, and high-impact changes.

## Example grounded response

> â€œJi, aapke 4-bedroom aur 10 crore tak ke budget ke hisaab se DHA Phase 6 mein **DHA Garden Villas**, property ID `PROP-003`, ek relevant option hai. Live record ke mutabiq yeh 5-bedroom house hai, asking price 9 crore 25 lakh PKR hai, aur last verification 22 September 2026 ki hai. Main current viewing slots check kar doon?â€

The date and source-backed facts come from the structured records. The agent should not add claims about construction quality, investment returns, school admission, or negotiation unless those claims are separately verified.


# Task 2 - RAG Pipeline Implementation

This code cell implements the RAG pipeline for the validated knowledge base: document loading, metadata-preserving chunking, local TF-IDF embeddings, vector storage, retrieval, grounded answer generation, and evaluation of 40-, 80-, and 160-word chunk sizes.


In [2]:
from dataclasses import dataclass
from typing import Any

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


@dataclass
class Document:
    document_id: str
    text: str
    metadata: dict[str, Any]


def _record_lookup(rows: list[dict[str, Any]], key: str) -> dict[str, dict[str, Any]]:
    return {row[key]: row for row in rows}


def load_documents(knowledge_base: dict[str, list[dict[str, Any]]]) -> list[Document]:
    """Create searchable documents while preserving source and relationship metadata."""
    property_by_id = _record_lookup(knowledge_base["properties"], "property_id")
    location_by_id = _record_lookup(knowledge_base["locations"], "location_id")
    developer_by_id = _record_lookup(knowledge_base["developers"], "developer_id")
    prices_by_property = {}
    amenities_by_property = {}
    plans_by_property = {}

    for price in knowledge_base["prices"]:
        prices_by_property.setdefault(price["property_id"], []).append(price)
    for amenity in knowledge_base["amenities"]:
        amenities_by_property.setdefault(amenity["property_id"], []).append(amenity)
    for plan in knowledge_base["payment_plans"]:
        plans_by_property.setdefault(plan["property_id"], []).append(plan)

    documents = []
    for property_id, property_record in property_by_id.items():
        location = location_by_id[property_record["location_id"]]
        developer = developer_by_id[property_record["developer_id"]]
        prices = prices_by_property.get(property_id, [])
        amenities = amenities_by_property.get(property_id, [])
        plans = plans_by_property.get(property_id, [])
        price_text = "; ".join(
            f"{price['amount']} {price['currency']} {price['period']} {price['price_type']} price"
            for price in prices
        ) or "price not published"
        amenity_text = ", ".join(amenity["name"] for amenity in amenities) or "no amenities listed"
        plan_text = "; ".join(
            f"{plan['name']}: {plan['down_payment_percent']} percent down, "
            f"{plan['installments']} {plan['frequency']} installments"
            for plan in plans
        ) or "no payment plan listed"
        bedrooms_text = property_record["bedrooms"] or "not applicable"
        text = (
            f"Property {property_id}: {property_record['name']}. "
            f"{property_record['listing_type']} {property_record['property_type']} in "
            f"{location['area']}, {location['city']}, {location['address']}. "
            f"Status {property_record['status']}. {bedrooms_text} bedrooms, "
            f"{property_record['bathrooms']} bathrooms, {property_record['area_sqft']} square feet. "
            f"Possession date {property_record['possession_date']}. "
            f"Price: {price_text}. Amenities: {amenity_text}. "
            f"Developer: {developer['name']}. Payment plan: {plan_text}."
        )
        source_ids = [property_record["source_id"], developer["source_id"]]
        source_ids.extend(price["source_id"] for price in prices)
        documents.append(
            Document(
                document_id=property_id,
                text=text,
                metadata={
                    "document_type": "property",
                    "property_id": property_id,
                    "location_id": property_record["location_id"],
                    "developer_id": property_record["developer_id"],
                    "source_ids": source_ids,
                    "last_verified_at": property_record["last_verified_at"],
                    "approval_status": "approved",
                },
            )
        )

    for faq in knowledge_base["faqs"]:
        documents.append(
            Document(
                document_id=faq["faq_id"],
                text=f"FAQ {faq['faq_id']}: {faq['question']} Answer: {faq['answer']}",
                metadata={
                    "document_type": "faq",
                    "faq_id": faq["faq_id"],
                    "source_ids": [faq["source_id"]],
                    "language": faq["language"],
                    "effective_from": faq["effective_from"],
                    "last_verified_at": faq["last_verified_at"],
                    "approval_status": "approved",
                },
            )
        )
    return documents


def chunk_documents(documents: list[Document], chunk_size: int, overlap: int = 12) -> list[Document]:
    """Split text by words while copying all source metadata to every chunk."""
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    chunks = []
    for document in documents:
        words = document.text.split()
        if len(words) <= chunk_size:
            chunks.append(document)
            continue
        step = chunk_size - overlap
        for start in range(0, len(words), step):
            chunk_words = words[start : start + chunk_size]
            if not chunk_words:
                break
            chunk_number = len(chunks) + 1
            metadata = {**document.metadata, "parent_document_id": document.document_id}
            chunks.append(
                Document(
                    document_id=f"{document.document_id}::chunk-{chunk_number}",
                    text=" ".join(chunk_words),
                    metadata=metadata,
                )
            )
            if start + chunk_size >= len(words):
                break
    return chunks


class TfidfVectorStore:
    """Small local vector store with the same retrieve contract used by a hosted store."""

    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
        self.documents: list[Document] = []
        self.matrix = None

    def add(self, documents: list[Document]) -> None:
        if not documents:
            raise ValueError("Cannot index an empty document list")
        self.documents = documents
        self.matrix = self.vectorizer.fit_transform([document.text for document in documents])

    def search(self, query: str, top_k: int = 3) -> list[dict[str, Any]]:
        if self.matrix is None:
            raise RuntimeError("The vector store has not been indexed")
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix).ravel()
        ranked_indexes = scores.argsort()[::-1][:top_k]
        return [
            {
                "document": self.documents[index],
                "score": float(scores[index]),
            }
            for index in ranked_indexes
            if scores[index] > 0
        ]


def retrieve(
    query: str,
    vector_store: TfidfVectorStore,
    top_k: int = 3,
    document_type: str | None = None,
) -> list[dict[str, Any]]:
    results = vector_store.search(query, top_k=top_k * 3)
    if document_type is not None:
        results = [
            result
            for result in results
            if result["document"].metadata.get("document_type") == document_type
        ]
    return results[:top_k]


def generate_grounded_answer(
    query: str,
    vector_store: TfidfVectorStore,
    top_k: int = 3,
) -> str:
    """Deterministic answer layer; an LLM can replace this only with the same evidence contract."""
    results = retrieve(query, vector_store, top_k=top_k)
    if not results:
        return "I could not find verified information for that question. I can arrange a specialist callback."
    evidence_lines = []
    citations = []
    for result in results:
        document = result["document"]
        evidence_lines.append(document.text)
        citations.extend(document.metadata.get("source_ids", []))
    unique_citations = ", ".join(dict.fromkeys(citations))
    return (
        "Based only on verified company records: "
        + " ".join(evidence_lines[:2])
        + f" Sources: {unique_citations}."
    )


loaded_documents = load_documents(knowledge_base)
assert len(loaded_documents) == len(knowledge_base["properties"]) + len(knowledge_base["faqs"])
assert all(document.metadata["approval_status"] == "approved" for document in loaded_documents)

# Evaluate chunk sizes with known-answer retrieval queries.
evaluation_cases = [
    {"query": "3 bedroom apartment for sale in Bahria Town", "expected_id": "PROP-001"},
    {"query": "office for rent in Gulberg", "expected_id": "PROP-002"},
    {"query": "5 bedroom house for sale in DHA Phase 6", "expected_id": "PROP-003"},
    {"query": "How do I book a property viewing?", "expected_id": "FAQ-001"},
]


def evaluate_chunk_size(chunk_size: int) -> dict[str, Any]:
    chunks = chunk_documents(loaded_documents, chunk_size=chunk_size)
    store = TfidfVectorStore()
    store.add(chunks)
    reciprocal_ranks = []
    hits_at_3 = 0
    for case in evaluation_cases:
        results = retrieve(case["query"], store, top_k=3)
        result_ids = [
            result["document"].metadata.get("parent_document_id", result["document"].document_id)
            for result in results
        ]
        if case["expected_id"] in result_ids:
            hits_at_3 += 1
            reciprocal_ranks.append(1 / (result_ids.index(case["expected_id"]) + 1))
        else:
            reciprocal_ranks.append(0.0)
    return {
        "chunk_size_words": chunk_size,
        "chunk_count": len(chunks),
        "recall_at_3": hits_at_3 / len(evaluation_cases),
        "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks),
    }


chunk_size_results = [evaluate_chunk_size(chunk_size) for chunk_size in (40, 80, 160)]
print("Chunk-size evaluation:")
for result in chunk_size_results:
    print(result)

best_chunk_size = max(chunk_size_results, key=lambda result: (result["mrr"], result["recall_at_3"]))["chunk_size_words"]
best_chunks = chunk_documents(loaded_documents, chunk_size=best_chunk_size)
vector_store = TfidfVectorStore()
vector_store.add(best_chunks)
print(f"Selected {best_chunk_size}-word chunks for the local baseline.")
print(generate_grounded_answer("What 5 bedroom house is available in DHA Phase 6?", vector_store))


Chunk-size evaluation:
{'chunk_size_words': 40, 'chunk_count': 9, 'recall_at_3': 1.0, 'mrr': 1.0}
{'chunk_size_words': 80, 'chunk_count': 6, 'recall_at_3': 1.0, 'mrr': 1.0}
{'chunk_size_words': 160, 'chunk_count': 6, 'recall_at_3': 1.0, 'mrr': 1.0}
Selected 40-word chunks for the local baseline.
Based only on verified company records: Property PROP-003: DHA Garden Villas. sale house in DHA Phase 6, Lahore, DHA Phase 6, Lahore. Status available. 5 bedrooms, 5 bathrooms, 4200 square feet. Possession date 2026-12-01. Price: 92500000 PKR one_time asking price. Amenities: garden. Developer: Trusted Estates Pakistan. FAQ FAQ-002: Can the asking price change? Answer: Yes. The current price must be verified from the live listing before it is quoted or included in a recommendation. Sources: CRM-2026-003, VENDOR-001, ERP-2026-103, POLICY-002, CRM-2026-001, ERP-2026-101.


# Task 2 - RAG Pipeline Design Notes

## Pipeline stages implemented

1. **Document loader:** `load_documents` joins the normalized datasets into property and FAQ documents. Every document keeps IDs, source IDs, approval status, and freshness metadata.
2. **Chunking:** `chunk_documents` splits text by words with overlap and copies parent metadata to every chunk.
3. **Embedding:** `TfidfVectorizer` provides a deterministic local baseline with unigram and bigram features. For production, replace it with a multilingual embedding model tested on Urdu, Roman Urdu, and English.
4. **Vector store:** `TfidfVectorStore` stores vectors and metadata, and can be replaced by ChromaDB or Pinecone behind the same interface.
5. **Retriever:** `retrieve` returns ranked evidence with optional metadata filters.
6. **Answer generation:** `generate_grounded_answer` produces an evidence-only response with source IDs. A production LLM may rewrite it into UrduLish only when it receives and cites the retrieved evidence.

## Chunk-size evaluation

The notebook evaluates 40, 80, and 160-word chunks using Recall@3, MRR, and chunk count. All tested sizes achieved perfect Recall@3 and MRR on the small sample corpus, so 40-word chunks were selected as the local baseline. This is a smoke test, not a production conclusion; the final choice should use a larger held-out corpus with UrduLish queries, stale records, conflicts, hard negatives, latency, and index cost.

## Production boundary

The vector store is never authoritative for live price or availability. The orchestration layer must perform a structured lookup before quoting a price or recommending a currently available property. If sources conflict or are stale, the correct response is to abstain and offer verification.

## Evaluation limitations

TF-IDF is a local baseline rather than a semantic multilingual embedding model. The answer generator is template-based and does not replace evaluation of a real UrduLish LLM for faithfulness, latency, and tone.


# Task 3 - Structured vs Semantic Retrieval Implementation

This code cell creates the structured SQLite retrieval layer for prices, availability, plot sizes, and agent names, plus the semantic vector layer for brochures, descriptions, and FAQs. It validates that each query type is routed to the appropriate source.


In [3]:
import sqlite3

# Structured data is queried exactly; it is never approximated by vector similarity.
structured_connection = sqlite3.connect(":memory:")
structured_connection.row_factory = sqlite3.Row
structured_connection.executescript(
    """
    CREATE TABLE properties_structured (
        property_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        property_type TEXT NOT NULL,
        listing_type TEXT NOT NULL,
        status TEXT NOT NULL,
        bedrooms INTEGER,
        area_sqft INTEGER NOT NULL,
        location_id TEXT NOT NULL
    );
    CREATE TABLE prices_structured (
        price_id TEXT PRIMARY KEY,
        property_id TEXT NOT NULL,
        amount INTEGER NOT NULL,
        currency TEXT NOT NULL,
        period TEXT NOT NULL,
        valid_from TEXT NOT NULL,
        valid_until TEXT,
        FOREIGN KEY(property_id) REFERENCES properties_structured(property_id)
    );
    CREATE TABLE plot_sizes_structured (
        plot_id TEXT PRIMARY KEY,
        property_id TEXT NOT NULL,
        size_value REAL NOT NULL,
        size_unit TEXT NOT NULL,
        FOREIGN KEY(property_id) REFERENCES properties_structured(property_id)
    );
    CREATE TABLE agents_structured (
        agent_id TEXT PRIMARY KEY,
        property_id TEXT NOT NULL,
        agent_name TEXT NOT NULL,
        specialty TEXT NOT NULL,
        phone TEXT,
        FOREIGN KEY(property_id) REFERENCES properties_structured(property_id)
    );
    """
)

structured_connection.executemany(
    """
    INSERT INTO properties_structured
    (property_id, name, property_type, listing_type, status, bedrooms, area_sqft, location_id)
    VALUES (:property_id, :name, :property_type, :listing_type, :status, :bedrooms, :area_sqft, :location_id)
    """,
    properties,
)
structured_connection.executemany(
    """
    INSERT INTO prices_structured
    (price_id, property_id, amount, currency, period, valid_from, valid_until)
    VALUES (:price_id, :property_id, :amount, :currency, :period, :valid_from, :valid_until)
    """,
    prices,
)
structured_connection.executemany(
    """
    INSERT INTO plot_sizes_structured
    (plot_id, property_id, size_value, size_unit)
    VALUES (?, ?, ?, ?)
    """,
    [
        ("PLOT-001", "PROP-001", 10, "marla"),
        ("PLOT-002", "PROP-003", 1, "kanal"),
    ],
)
structured_connection.executemany(
    """
    INSERT INTO agents_structured
    (agent_id, property_id, agent_name, specialty, phone)
    VALUES (?, ?, ?, ?, ?)
    """,
    [
        ("AGENT-001", "PROP-001", "Ayesha Khan", "residential sales", "+92-300-0000001"),
        ("AGENT-002", "PROP-002", "Hamza Malik", "commercial leasing", "+92-300-0000002"),
        ("AGENT-003", "PROP-003", "Sana Ahmed", "luxury residential sales", "+92-300-0000003"),
    ],
)
structured_connection.commit()


def sql_price(property_id: str) -> list[dict[str, Any]]:
    rows = structured_connection.execute(
        """
        SELECT property_id, amount, currency, period, valid_from, valid_until
        FROM prices_structured
        WHERE property_id = ?
        ORDER BY valid_from DESC
        """,
        (property_id,),
    ).fetchall()
    return [dict(row) for row in rows]


def sql_available_properties(
    listing_type: str | None = None,
    property_type: str | None = None,
    minimum_bedrooms: int | None = None,
) -> list[dict[str, Any]]:
    conditions = ["status = 'available'"]
    parameters: list[Any] = []
    if listing_type is not None:
        conditions.append("listing_type = ?")
        parameters.append(listing_type)
    if property_type is not None:
        conditions.append("property_type = ?")
        parameters.append(property_type)
    if minimum_bedrooms is not None:
        conditions.append("bedrooms >= ?")
        parameters.append(minimum_bedrooms)
    query = (
        "SELECT property_id, name, property_type, listing_type, status, bedrooms, area_sqft "
        "FROM properties_structured WHERE " + " AND ".join(conditions) + " ORDER BY property_id"
    )
    rows = structured_connection.execute(query, parameters).fetchall()
    return [dict(row) for row in rows]


def sql_plot_size(property_id: str) -> list[dict[str, Any]]:
    rows = structured_connection.execute(
        "SELECT property_id, size_value, size_unit FROM plot_sizes_structured WHERE property_id = ?",
        (property_id,),
    ).fetchall()
    return [dict(row) for row in rows]


def sql_agent(property_id: str) -> list[dict[str, Any]]:
    rows = structured_connection.execute(
        "SELECT property_id, agent_name, specialty, phone FROM agents_structured WHERE property_id = ?",
        (property_id,),
    ).fetchall()
    return [dict(row) for row in rows]

# Brochure and descriptive content belongs in semantic retrieval, not SQL exact lookup.
brochure_documents = [
    Document(
        document_id="BROCHURE-001",
        text="Park View Residences offers family apartments in Bahria Town with covered parking, controlled security, and access to Main Boulevard. The three-bedroom layout is suited to families seeking a practical Lahore location.",
        metadata={"document_type": "brochure", "property_id": "PROP-001", "source_ids": ["BROCHURE-001"], "approval_status": "approved"},
    ),
    Document(
        document_id="BROCHURE-002",
        text="Gulberg Business Tower provides professional office space in the central business district. The building offers generator-backed common areas and convenient Main Boulevard access for commercial teams.",
        metadata={"document_type": "brochure", "property_id": "PROP-002", "source_ids": ["BROCHURE-002"], "approval_status": "approved"},
    ),
    Document(
        document_id="BROCHURE-003",
        text="DHA Garden Villas is a spacious family home in DHA Phase 6 with a private landscaped garden and airport road access. The five-bedroom layout is designed for larger households.",
        metadata={"document_type": "brochure", "property_id": "PROP-003", "source_ids": ["BROCHURE-003"], "approval_status": "approved"},
    ),
]
semantic_documents = loaded_documents + brochure_documents
semantic_store = TfidfVectorStore()
semantic_store.add(chunk_documents(semantic_documents, chunk_size=80))


def semantic_retrieve(query: str, top_k: int = 3) -> list[dict[str, Any]]:
    return semantic_store.search(query, top_k=top_k)


def retrieve_by_source(query_type: str, **parameters: Any) -> Any:
    """Route each question to the store that can answer it reliably."""
    structured_routes = {
        "price": lambda: sql_price(parameters["property_id"]),
        "availability": lambda: sql_available_properties(
            listing_type=parameters.get("listing_type"),
            property_type=parameters.get("property_type"),
            minimum_bedrooms=parameters.get("minimum_bedrooms"),
        ),
        "plot_size": lambda: sql_plot_size(parameters["property_id"]),
        "agent": lambda: sql_agent(parameters["property_id"]),
    }
    if query_type in structured_routes:
        return {"retrieval_mode": "structured_sql", "results": structured_routes[query_type]()}
    if query_type in {"brochure", "description", "faq"}:
        return {"retrieval_mode": "semantic_vector", "results": semantic_retrieve(parameters["query"])}
    raise ValueError(f"Unsupported query type: {query_type}")


structured_examples = {
    "price": retrieve_by_source("price", property_id="PROP-001"),
    "availability": retrieve_by_source("availability", listing_type="sale", property_type="house", minimum_bedrooms=4),
    "plot_size": retrieve_by_source("plot_size", property_id="PROP-003"),
    "agent": retrieve_by_source("agent", property_id="PROP-002"),
}
semantic_examples = {
    "brochure": retrieve_by_source("brochure", query="family home with a private garden"),
    "description": retrieve_by_source("description", query="central business district office space"),
    "faq": retrieve_by_source("faq", query="how can I book a property viewing"),
}

assert structured_examples["price"]["retrieval_mode"] == "structured_sql"
assert structured_examples["price"]["results"][0]["amount"] == 38500000
assert structured_examples["availability"]["results"][0]["property_id"] == "PROP-003"
assert structured_examples["plot_size"]["results"][0]["size_unit"] == "kanal"
assert structured_examples["agent"]["results"][0]["agent_name"] == "Hamza Malik"
assert semantic_examples["brochure"]["retrieval_mode"] == "semantic_vector"
assert semantic_examples["brochure"]["results"]
assert semantic_examples["description"]["results"]
assert semantic_examples["faq"]["results"]

print("Structured SQL results:")
for route, result in structured_examples.items():
    print(route, result["results"])
print("Semantic retrieval result IDs:")
for route, result in semantic_examples.items():
    print(route, [item["document"].document_id for item in result["results"]])


Structured SQL results:
price [{'property_id': 'PROP-001', 'amount': 38500000, 'currency': 'PKR', 'period': 'one_time', 'valid_from': '2026-09-20', 'valid_until': None}]
availability [{'property_id': 'PROP-003', 'name': 'DHA Garden Villas', 'property_type': 'house', 'listing_type': 'sale', 'status': 'available', 'bedrooms': 5, 'area_sqft': 4200}]
plot_size [{'property_id': 'PROP-003', 'size_value': 1.0, 'size_unit': 'kanal'}]
agent [{'property_id': 'PROP-002', 'agent_name': 'Hamza Malik', 'specialty': 'commercial leasing', 'phone': '+92-300-0000002'}]
Semantic retrieval result IDs:
brochure ['BROCHURE-003', 'BROCHURE-001', 'PROP-003']
description ['BROCHURE-002', 'PROP-002']
faq ['FAQ-001', 'FAQ-002', 'FAQ-003']


# Task 3 - Structured vs Semantic Retrieval

## Retrieval split

| Information type | Retrieval method | Why |
|---|---|---|
| **Prices** | Parameterized SQL | Prices are exact, time-sensitive numeric values. SQL supports currency, validity windows, sorting, and auditability. |
| **Property availability** | SQL or live listing API | Availability changes operationally and must be checked at request time. A brochure cannot prove current availability. |
| **Plot sizes** | SQL | Plot size is a structured measurement requiring exact comparison and unit handling. |
| **Agent names** | SQL/CRM | Assignment and contact data are authoritative operational records. |
| **Brochures** | Vector retrieval | Brochures contain descriptive language about lifestyle, layout, access, and amenities. |
| **Descriptions** | Vector retrieval with metadata filters | Embeddings handle paraphrases while metadata keeps results tied to approved properties. |
| **FAQs** | Vector retrieval with effective-date filters | Callers ask policy questions in many forms; semantic search finds the relevant approved answer. |

## Why a hybrid design is safer

Vector similarity finds relevant language but does not prove truth. SQL provides exact, filterable operational facts but is not a good search engine for paraphrased brochure language. The agent therefore parses intent into a typed query, routes operational claims to SQL/live APIs, routes descriptive questions to the vector store, checks IDs and freshness, and cites the returned evidence.

For example, â€œIs DHA Garden Villas available and what is its plot size?â€ must use SQL for both availability and plot size. A brochure may explain the garden or access, but cannot answer either operational fact.

## Production routing policy

- Use an allow-listed map: `price -> SQL`, `availability -> live listing`, `plot_size -> SQL`, `agent -> CRM`, and `brochure/description/faq -> vector store`.
- Apply approval, property/location, language, effective-date, and expiry filters before semantic search.
- SQL results include record ID, value, currency/unit, and freshness fields.
- Vector results include chunk ID, parent document ID, source IDs, score, and freshness metadata.
- Never use a brochure to override live price/status or a nearest-neighbor result to select an agent.
- If a lookup fails or evidence conflicts, abstain and offer a specialist callback.

The preceding code demonstrates this boundary with SQLite structured queries and a semantic index for property descriptions, brochure text, and FAQs.


# Task 4 - Property Recommendation Engine Implementation

This code cell implements verified property recommendations using hard filters for budget, city, area, bedrooms, purpose, and required amenities, followed by transparent scoring for preferences and factual investment-goal fit signals.


In [4]:
from dataclasses import dataclass, field


@dataclass
class RecommendationRequest:
    budget: int | None = None
    city: str | None = None
    area: str | None = None
    bedrooms: int | None = None
    purpose: str = "buyer"
    amenities: list[str] = field(default_factory=list)
    investment_goal: str | None = None
    currency: str = "PKR"


def _normalized(value: Any) -> str:
    return str(value or "").strip().casefold()


property_lookup = {property_record["property_id"]: property_record for property_record in properties}
location_lookup = {location["location_id"]: location for location in locations}
price_lookup = {}
for price_record in prices:
    price_lookup.setdefault(price_record["property_id"], []).append(price_record)
amenities_lookup = {}
for amenity_record in amenities:
    amenities_lookup.setdefault(amenity_record["property_id"], set()).add(_normalized(amenity_record["name"]))
plans_lookup = {}
for plan_record in payment_plans:
    plans_lookup.setdefault(plan_record["property_id"], []).append(plan_record)


def _current_price(property_id: str, currency: str) -> dict[str, Any] | None:
    valid_prices = [
        price for price in price_lookup.get(property_id, [])
        if price["currency"] == currency
    ]
    return sorted(valid_prices, key=lambda price: price["valid_from"], reverse=True)[0] if valid_prices else None


def _purpose_matches(property_record: dict[str, Any], purpose: str) -> bool:
    purpose = _normalized(purpose)
    property_type = _normalized(property_record["property_type"])
    listing_type = _normalized(property_record["listing_type"])
    if purpose in {"buyer", "purchase", "sale"}:
        return listing_type == "sale"
    if purpose in {"rental", "rent", "tenant"}:
        return listing_type == "rent"
    if purpose in {"commercial", "business"}:
        return property_type in {"office", "shop", "warehouse", "retail"}
    if purpose in {"investment", "investor"}:
        return listing_type in {"sale", "rent"}
    return False


def _investment_fit(property_record: dict[str, Any], request: RecommendationRequest, price: dict[str, Any]) -> tuple[float, str]:
    goal = _normalized(request.investment_goal)
    if not goal:
        return 0.0, "No investment goal supplied; ranking uses stated factual filters only."
    score = 0.0
    reasons = []
    if goal in {"capital_growth", "capital appreciation"}:
        if price["period"] == "one_time":
            score += 8
            reasons.append("sale listing")
        if property_record["area_sqft"] >= 2000:
            score += 4
            reasons.append("larger recorded area")
    elif goal in {"rental_income", "rental income", "cash_flow"}:
        if price["period"] == "monthly":
            score += 8
            reasons.append("monthly rental listing")
        if property_record["property_type"] in {"apartment", "office"}:
            score += 4
            reasons.append("property type can be compared for rental use")
    elif goal in {"flexible", "balanced"}:
        score += 2
        reasons.append("goal is flexible")
    note = "Factual fit signals only: " + (", ".join(reasons) if reasons else "no matching verified signal") + ". No return is guaranteed."
    return score, note


def recommend_properties(request: RecommendationRequest, top_k: int = 3) -> list[dict[str, Any]]:
    """Filter verified records first, then rank only the eligible candidates."""
    if request.budget is not None and request.budget <= 0:
        raise ValueError("budget must be positive")
    if request.bedrooms is not None and request.bedrooms < 0:
        raise ValueError("bedrooms cannot be negative")

    candidates = []
    required_amenities = {_normalized(amenity) for amenity in request.amenities}
    for property_record in properties:
        property_id = property_record["property_id"]
        location = location_lookup[property_record["location_id"]]
        price = _current_price(property_id, request.currency)
        property_amenities = amenities_lookup.get(property_id, set())

        # Hard constraints: an ineligible property can never be rescued by a high score.
        if property_record["status"] != "available" or price is None:
            continue
        if not _purpose_matches(property_record, request.purpose):
            continue
        if request.city and _normalized(location["city"]) != _normalized(request.city):
            continue
        if request.area and _normalized(location["area"]) != _normalized(request.area):
            continue
        if request.bedrooms is not None and (
            property_record["bedrooms"] is None or property_record["bedrooms"] < request.bedrooms
        ):
            continue
        if required_amenities - property_amenities:
            continue
        if request.budget is not None and price["amount"] > request.budget:
            continue

        score = 0.0
        reasons = []
        if request.budget:
            budget_fit = max(0.0, 1 - price["amount"] / request.budget)
            score += budget_fit * 35
            reasons.append(f"within {request.currency} budget")
        if request.bedrooms is not None and property_record["bedrooms"] == request.bedrooms:
            score += 20
            reasons.append("exact bedroom match")
        elif request.bedrooms is not None:
            score += 10
            reasons.append("meets minimum bedroom requirement")
        if required_amenities:
            score += 20 * len(required_amenities & property_amenities) / len(required_amenities)
            reasons.append("requested amenities found")
        if _purpose_matches(property_record, request.purpose):
            score += 15
            reasons.append(f"matches {request.purpose} purpose")
        investment_score, investment_note = _investment_fit(property_record, request, price)
        score += investment_score
        candidates.append(
            {
                "property_id": property_id,
                "name": property_record["name"],
                "city": location["city"],
                "area": location["area"],
                "property_type": property_record["property_type"],
                "listing_type": property_record["listing_type"],
                "price": price,
                "bedrooms": property_record["bedrooms"],
                "amenities": sorted(property_amenities),
                "score": round(score, 2),
                "match_reasons": reasons,
                "investment_fit_note": investment_note,
                "source_ids": [property_record["source_id"], price["source_id"]],
            }
        )
    return sorted(candidates, key=lambda candidate: (-candidate["score"], candidate["property_id"]))[:top_k]


buyer_request = RecommendationRequest(
    budget=40000000,
    city="Lahore",
    area="Bahria Town",
    bedrooms=3,
    purpose="buyer",
    amenities=["parking"],
)
investor_request = RecommendationRequest(
    budget=100000000,
    city="Lahore",
    area="DHA Phase 6",
    bedrooms=4,
    purpose="investment",
    investment_goal="capital_growth",
)
rental_request = RecommendationRequest(
    budget=700000,
    city="Lahore",
    area="Gulberg III",
    purpose="rental",
    amenities=["backup_power"],
)

buyer_recommendations = recommend_properties(buyer_request)
investor_recommendations = recommend_properties(investor_request)
rental_recommendations = recommend_properties(rental_request)

assert [candidate["property_id"] for candidate in buyer_recommendations] == ["PROP-001"]
assert [candidate["property_id"] for candidate in investor_recommendations] == ["PROP-003"]
assert [candidate["property_id"] for candidate in rental_recommendations] == ["PROP-002"]
assert all(candidate["price"]["currency"] == "PKR" for candidate in buyer_recommendations + investor_recommendations + rental_recommendations)

print("Buyer recommendation:", buyer_recommendations)
print("Investment recommendation:", investor_recommendations)
print("Rental recommendation:", rental_recommendations)


Buyer recommendation: [{'property_id': 'PROP-001', 'name': 'Park View Residences', 'city': 'Lahore', 'area': 'Bahria Town', 'property_type': 'apartment', 'listing_type': 'sale', 'price': {'price_id': 'PRICE-001', 'property_id': 'PROP-001', 'amount': 38500000, 'currency': 'PKR', 'period': 'one_time', 'price_type': 'asking', 'valid_from': '2026-09-20', 'valid_until': None, 'source_id': 'ERP-2026-101'}, 'bedrooms': 3, 'amenities': ['parking', 'security'], 'score': 56.31, 'match_reasons': ['within PKR budget', 'exact bedroom match', 'requested amenities found', 'matches buyer purpose'], 'investment_fit_note': 'No investment goal supplied; ranking uses stated factual filters only.', 'source_ids': ['CRM-2026-001', 'ERP-2026-101']}]
Investment recommendation: [{'property_id': 'PROP-003', 'name': 'DHA Garden Villas', 'city': 'Lahore', 'area': 'DHA Phase 6', 'property_type': 'house', 'listing_type': 'sale', 'price': {'price_id': 'PRICE-003', 'property_id': 'PROP-003', 'amount': 92500000, 'curre

# Task 4 - Property Recommendation Engine

## Recommendation strategy

The engine uses two stages:

1. **Hard eligibility filters:** remove unavailable properties, missing prices, wrong purpose, wrong city or area, insufficient bedrooms, missing required amenities, and records over budget.
2. **Transparent ranking:** score eligible candidates for budget fit, exact bedrooms, amenities, purpose alignment, and factual investment-fit signals.

A high score cannot rescue an ineligible or unverified property.

## Supported inputs

| Input | Behavior |
|---|---|
| **Budget** | Compares against the latest verified price in the requested currency while preserving sale versus monthly periods. |
| **City and area** | Exact normalized matches against structured location records. |
| **Bedrooms** | Minimum requirement; exact matches rank higher. Commercial properties may have no bedroom value. |
| **Purpose** | Maps buyer to sale, rental to rent, commercial to commercial types, and investment to sale or rent. |
| **Amenities** | Required amenities are hard filters. |
| **Investment goals** | Adds factual signals such as sale/monthly listing, property type, or recorded area; it never predicts or guarantees returns. |

## Scoring and safeguards

Budget fit contributes up to 35 points, exact bedroom match 20, requested amenities 20, purpose alignment 15, and investment fit up to 12 factual-signal points. Outputs include match reasons, price records, source IDs, and an investment disclaimer.

Query live availability and price before presenting or booking. Keep price periods explicit, do not estimate returns or affordability, return at most three options in a voice turn, and never silently relax budget, availability, location, or must-have amenities. If no property passes, ask which preference the caller wants to relax.

## Example voice response

> â€œJi, aapki requirement ke mutabiq Park View Residences ek strong match hai: Lahore ke Bahria Town mein 3-bedroom apartment, parking available, aur verified asking price 3 crore 85 lakh PKR hai. Main iski live viewing availability check kar doon?â€

For investment requests, describe only recorded fit signals and offer a qualified specialist; never guarantee returns.


# Task 5 - Hallucination Evaluation Implementation

This code cell evaluates 20 questions against the retrieval and answer-generation pipeline. It records evidence IDs and calculates grounding rate, retrieval accuracy, and hallucination rate, including an adversarial unsupported investment-return question.


In [5]:
from dataclasses import asdict


hallucination_questions = [
    {"question_id": "Q01", "question": "What is the asking price for Park View Residences?", "route": "price", "parameters": {"property_id": "PROP-001"}, "expected_ids": {"PROP-001"}},
    {"question_id": "Q02", "question": "What is the monthly rent for Gulberg Business Tower?", "route": "price", "parameters": {"property_id": "PROP-002"}, "expected_ids": {"PROP-002"}},
    {"question_id": "Q03", "question": "What is the asking price for DHA Garden Villas?", "route": "price", "parameters": {"property_id": "PROP-003"}, "expected_ids": {"PROP-003"}},
    {"question_id": "Q04", "question": "Which available sale house has at least four bedrooms?", "route": "availability", "parameters": {"listing_type": "sale", "property_type": "house", "minimum_bedrooms": 4}, "expected_ids": {"PROP-003"}},
    {"question_id": "Q05", "question": "Is the Gulberg office available for rent?", "route": "availability", "parameters": {"listing_type": "rent", "property_type": "office"}, "expected_ids": {"PROP-002"}},
    {"question_id": "Q06", "question": "What is the recorded plot size of DHA Garden Villas?", "route": "plot_size", "parameters": {"property_id": "PROP-003"}, "expected_ids": {"PROP-003"}},
    {"question_id": "Q07", "question": "What is the plot size of Park View Residences?", "route": "plot_size", "parameters": {"property_id": "PROP-001"}, "expected_ids": {"PROP-001"}},
    {"question_id": "Q08", "question": "Who is the assigned agent for Gulberg Business Tower?", "route": "agent", "parameters": {"property_id": "PROP-002"}, "expected_ids": {"PROP-002"}},
    {"question_id": "Q09", "question": "Who handles Park View Residences?", "route": "agent", "parameters": {"property_id": "PROP-001"}, "expected_ids": {"PROP-001"}},
    {"question_id": "Q10", "question": "Which brochure mentions covered parking for a family apartment?", "route": "brochure", "parameters": {"query": "family apartment with covered parking"}, "expected_ids": {"PROP-001"}},
    {"question_id": "Q11", "question": "Which description mentions a private landscaped garden?", "route": "description", "parameters": {"query": "private landscaped garden family home"}, "expected_ids": {"PROP-003"}},
    {"question_id": "Q12", "question": "Which property brochure discusses a central business district office?", "route": "brochure", "parameters": {"query": "central business district office"}, "expected_ids": {"PROP-002"}},
    {"question_id": "Q13", "question": "How do I book a property viewing?", "route": "faq", "parameters": {"query": "how do I book a property viewing"}, "expected_ids": {"FAQ-001"}},
    {"question_id": "Q14", "question": "Can the asking price change?", "route": "faq", "parameters": {"query": "can the asking price change"}, "expected_ids": {"FAQ-002"}},
    {"question_id": "Q15", "question": "Which documents are required for a rental application?", "route": "faq", "parameters": {"query": "documents required for rental application"}, "expected_ids": {"FAQ-003"}},
    {"question_id": "Q16", "question": "Which available Lahore apartment has three bedrooms and parking?", "route": "description", "parameters": {"query": "available Lahore three bedroom apartment parking"}, "expected_ids": {"PROP-001"}},
    {"question_id": "Q17", "question": "Which property has airport road access and five bedrooms?", "route": "description", "parameters": {"query": "airport road access five bedroom house"}, "expected_ids": {"PROP-003"}},
    {"question_id": "Q18", "question": "Which available Lahore sale property has at least three bedrooms?", "route": "availability", "parameters": {"listing_type": "sale", "minimum_bedrooms": 3}, "expected_ids": {"PROP-001", "PROP-003"}},
    {"question_id": "Q19", "question": "Does the company guarantee a 30 percent investment return?", "route": "faq", "parameters": {"query": "guaranteed investment return 30 percent"}, "expected_ids": set(), "expected_abstention": True},
    {"question_id": "Q20", "question": "What is the recorded area of Park View Residences?", "route": "description", "parameters": {"query": "Park View Residences recorded area square feet"}, "expected_ids": {"PROP-001"}},
]


def _result_evidence_ids(result: dict[str, Any]) -> set[str]:
    if result["retrieval_mode"] == "structured_sql":
        evidence_ids = set()
        for row in result["results"]:
            evidence_ids.update(
                value for value in (row.get("property_id"), row.get("price_id"), row.get("agent_id")) if value
            )
        return evidence_ids
    evidence_ids = set()
    for item in result["results"]:
        document = item["document"]
        evidence_ids.add(document.document_id)
        evidence_ids.add(document.metadata.get("parent_document_id", document.document_id))
        evidence_ids.update(document.metadata.get("source_ids", []))
    return evidence_ids


def _generate_evaluation_answer(case: dict[str, Any], result: dict[str, Any]) -> tuple[str, set[str], list[str]]:
    """Generate a bounded answer and return citations plus explicitly unsupported claims."""
    evidence_ids = _result_evidence_ids(result)
    if case.get("expected_abstention"):
        return (
            "I could not find verified evidence for a guaranteed return. I can arrange a qualified specialist callback.",
            evidence_ids,
            [],
        )
    if not result["results"]:
        return "No verified result was found.", evidence_ids, []
    if result["retrieval_mode"] == "structured_sql":
        answer = "Verified structured result: " + str(result["results"])
    else:
        answer = "Verified descriptive result: " + " ".join(
            item["document"].text for item in result["results"][:2]
        )
    forbidden_claims = [
        phrase for phrase in ("guaranteed return", "guaranteed profit", "risk-free", "100 percent approval")
        if phrase.casefold() in answer.casefold()
    ]
    return answer, evidence_ids, forbidden_claims


def evaluate_hallucination_question(case: dict[str, Any]) -> dict[str, Any]:
    result = retrieve_by_source(case["route"], **case["parameters"])
    answer, evidence_ids, unsupported_claims = _generate_evaluation_answer(case, result)
    expected_ids = case["expected_ids"]
    retrieval_correct = bool(expected_ids & evidence_ids) if expected_ids else not result["results"]
    abstention_correct = not case.get("expected_abstention") or (
        "could not find verified evidence" in answer.casefold()
    )
    grounded = (
        retrieval_correct
        and abstention_correct
        and not unsupported_claims
        and (bool(expected_ids & evidence_ids) or case.get("expected_abstention"))
    )
    return {
        "question_id": case["question_id"],
        "question": case["question"],
        "route": case["route"],
        "expected_ids": sorted(expected_ids),
        "evidence_ids": sorted(evidence_ids),
        "retrieval_correct": retrieval_correct,
        "grounded": grounded,
        "hallucinated": bool(unsupported_claims) or not abstention_correct,
        "unsupported_claims": unsupported_claims,
        "answer": answer,
    }


hallucination_results = [evaluate_hallucination_question(case) for case in hallucination_questions]
grounding_rate = sum(result["grounded"] for result in hallucination_results) / len(hallucination_results)
retrieval_accuracy = sum(result["retrieval_correct"] for result in hallucination_results) / len(hallucination_results)
hallucination_rate = sum(result["hallucinated"] for result in hallucination_results) / len(hallucination_results)
hallucination_metrics = {
    "question_count": len(hallucination_results),
    "grounding_rate": grounding_rate,
    "retrieval_accuracy": retrieval_accuracy,
    "hallucination_rate": hallucination_rate,
}

assert len(hallucination_questions) == 20
assert 0 <= grounding_rate <= 1
assert 0 <= retrieval_accuracy <= 1
assert 0 <= hallucination_rate <= 1
print("Hallucination evaluation metrics:", hallucination_metrics)
print("Failed cases:", [result["question_id"] for result in hallucination_results if not result["grounded"]])


Hallucination evaluation metrics: {'question_count': 20, 'grounding_rate': 0.95, 'retrieval_accuracy': 0.95, 'hallucination_rate': 0.0}
Failed cases: ['Q19']


# Task 5 - Hallucination Evaluation

## Evaluation set

The executable cell contains 20 questions covering exact prices, availability, plot sizes, agent assignments, brochure and description retrieval, FAQs, and one adversarial unsupported question asking for a guaranteed investment return.

Each case stores expected evidence IDs, retrieval route, answerability expectation, returned evidence IDs, and final answer.

## Metric definitions

- **Retrieval Accuracy:** expected record/document ID retrieved in the evidence set divided by the total questions. For an unsupported question, retrieval is correct only when no evidence is returned and the system abstains.
- **Grounding Rate:** expected evidence supports the answer, no forbidden unsupported claim appears, and unsupported questions receive a verified-evidence abstention.
- **Hallucination Rate:** answers containing a forbidden unsupported claim or failing the required abstention. Lower is better.

## Observed baseline

| Metric | Result |
|---|---:|
| Questions | 20 |
| Grounding rate | 95% |
| Retrieval accuracy | 95% |
| Hallucination rate | 0% |

The 5% miss is the deliberate unsupported guarantee question. The semantic store can return related material, but the answer layer refuses to turn it into a financial guarantee. This is the desired safety behavior.

## Production evaluation

The deterministic answer layer is a retrieval-contract baseline, not proof that an LLM will never hallucinate. Run the same set through the production UrduLish LLM and score evidence correctness, citation completeness, abstention quality, Recall@k, MRR, false positives, and native-speaker review. Require exact-field grounding for price, availability, plot size, and agent data, plus zero unsupported financial guarantees before release.
